## WormLib: 3D Co-localization Analysis Module

In [ ]:
# 6. Colocalization calculations
export CALCULATE_NUCLEI_COLOCALIZATION="False"
export GENERATE_DONUT_MASK="False"
export CALCULATE_MEMBRANE_COLOCALIZATION="False"
export RUN_CONCENTRIC_LAYERS_ANALYSIS="False"
export GENERATE_PGRANULE_MASK="False"
export CALCULATE_PGRANULE_COLOCALIZATION="False"
export CALCULATE_mRNA_mRNA_COLOCALIZATION="False"
export RUN_PROTEIN_HEATMAPS="False"


In [ ]:
# #Install packages

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from skimage.morphology import binary_erosion, disk

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy.ndimage import zoom




In [ ]:
# 6. Colocalization analysis module
calculate_nuclei_colocalization = False

generate_donut_mask = False
calculate_membrane_colocalization = False
run_concentric_layers_analysis = False

generate_pgranule_mask = False
calculate_pgranule_colocalization = False

calculate_mRNA_mRNA_colocalization = False
run_protein_heatmaps = False

In [ ]:
# #Tidy up before plotting

# Clean arrays to calculate special features in 3D
ch0_array = spots_post_clustering_ch0
# Check if the array has exactly 4 columns
if ch0_array.shape[1] == 4 :
    # Remove the last column
    ch0_array = ch0_array[:, :-1]
#     print("Last column removed.")
else:
    # Keep the array unchanged
    ch0_array  = ch0_array
    print("Array unchanged.")

#Display the cleaned array
# print("Cleaned ch0 array:")
# print(ch0_array)

print("\n")

# Original array (example)
ch1_array = spots_post_clustering_ch1
# Check if the array has exactly 4 columns 
if ch1_array.shape[1] == 4 :
    # Remove the last column
    ch1_array = ch1_array[:, :-1]
#     print("Last column removed.")
else:
    # Keep the array unchanged
    ch1_array = ch1_array
    print("Array unchanged.")

#Display the cleaned array
# print("Cleaned ch1 array:")
# print(ch1_array)


### 1. Nuclear colocalization analysis

In [ ]:
# #Calculate nuclear colocalization

# masks_nuclei.shape # mask nuclei is a 2D array that contains the 3D data
nuc_max = np.max(image_colors[1, :, :, :], axis=0)  # mCherry channel

if calculate_nuclei_colocalization:
    def calculate_nuclei_colocalization(coord, rna_max, masks_nuclei, output_directory, rna_channel, image_name, plot=True):
        """
        Calculate and optionally plot the colocalization of RNA spots with nuclei.

        Parameters:
            coord: np.ndarray
                The coordinates of RNA spots.
            rna_max: np.ndarray
                Maximum projection of the RNA channel for plotting.
            masks_nuclei: np.ndarray
                Mask of the nuclei region.
            output_directory: str
                Directory to save the output plots.
            rna_channel: str
                Name of the RNA channel (e.g., "mCherry").
            image_name: str
                Name of the image being processed.
            plot: bool
                Whether to create and save plots.

        Returns:
            tuple: (spots_in_nuclei, spots_out_nuclei)
        """
        ndim = 3  # 3D data

        # Identify objects in nuclei
        spots_in_nuclei, spots_out_nuclei = bigfish.multistack.identify_objects_in_region(masks_nuclei, coord, ndim)

        # Log the results
        print(f'{rna_channel} spots detected (in nuclei):')
        print(f"  Shape: {spots_in_nuclei.shape}")
        print(f"  Dtype: {spots_in_nuclei.dtype}\n")
        print(f'{rna_channel} spots detected (in cytoplasm):')
        print(f"  Shape: {spots_out_nuclei.shape}")
        print(f"  Dtype: {spots_out_nuclei.dtype}")

        if plot:
            # Plot spots in cytoplasm
            out_nuclei_output = os.path.join(output_directory, rna_channel + '_in_cyto_' + image_name + '.png')
            plot.plot_detection(
                rna_max,
                spots=spots_out_nuclei,
                radius=1,
                color="blue",
                path_output=out_nuclei_output,
                title=f'Blue = {rna_channel} in cytoplasm',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

            # Plot spots in nuclei
            in_nuclei_output = os.path.join(output_directory, rna_channel + '_in_nuclei_' + image_name + '.png')
            plot.plot_detection(
                masks_nuclei,
                spots=spots_in_nuclei,
                radius=1,
                color="red",
                path_output=in_nuclei_output,
                title=f'Red = {rna_channel} in nuclei',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

            # Combined plot
            combined_output = os.path.join(output_directory, rna_channel + '_combined_nuclei_' + image_name + '.png')
            plot.plot_detection(
                rna_max,
                spots=[spots_in_nuclei, spots_out_nuclei],
                radius=2,
                color=["red", "blue"],
                path_output=combined_output,
                title=f'Red = {rna_channel} in nuclei | Blue = {rna_channel} in cytoplasm',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

        # Quantification: number of spots in nuclei and cytoplasm
        num_spots_in_nuclei = spots_in_nuclei.shape[0]
        # num_spots_in_cyto = spots_out_nuclei.shape[0]

        # Add new columns with default values (e.g., NaN)
        df_quantification[f'{rna_channel} in nuclei'] = np.nan
        # df_quantification[f'{rna_channel} out nuclei'] = np.nan

        # Update the new columns with the counts for the specific image ID
        df_quantification.loc[df_quantification['Image ID'] == image_name, f'{rna_channel} in nuclei'] = num_spots_in_nuclei
        # df_quantification.loc[df_quantification['Image ID'] == image_name, f'{rna_channel} out nuclei'] = num_spots_in_cyto

        # Display the updated DataFrame
        df_quantification

        return spots_in_nuclei, spots_out_nuclei
    
    # Call the function for both RNA channels
    rna_names = [Cy5, mCherry]  # Replace with your actual mRNA names
    rna_images = [image_Cy5, image_mCherry]  # Replace with your actual 2D/3D RNA images
    rna_coords = [ch0_array, ch1_array]  # Coordinates for each RNA channel
    

    # Call the function for each RNA channel
    for rna_name, rna_image, coord in zip(rna_names, rna_images, rna_coords):
        spots_in_nuclei, spots_out_nuclei = calculate_nuclei_colocalization(
            coord=coord,
            rna_max=rna_image,
            masks_nuclei=masks_nuclei,
            output_directory=output_directory,
            rna_channel=rna_name,
            image_name=image_name,
            plot=plot
        )
# Display the updated DataFrame
df_quantification

### 2. Membrane colocalization analysis

#### 2.1 Generate donut masks

In [ ]:
# #Generate donut masks to calculate membrane colocalization
if generate_donut_mask:
    def generate_donut_mask(original_mask, n, plot=False, output_path=None):
        # Initialize the final mask
        donut_mask = np.zeros_like(original_mask)

        for i in np.unique(original_mask):
            if i == 0:
                continue  # Skip the background (assuming it's labeled as 0)

            selected_cel = original_mask == i

            # Erode the original mask by n pixels using a disk-shaped structuring element
            selem = disk(n)
            eroded_mask = binary_erosion(selected_cel, selem)

            # Create the border mask by subtracting the eroded mask from the original
            donut = selected_cel & ~eroded_mask
            donut_mask[donut] = i

        # Plotting the border mask if 'plot' is set to True
        if plot:
            plt.figure(figsize=(5, 5))
            plt.imshow(donut_mask, cmap='Greys_r')
            plt.title(f'Donut Mask')
            plt.axis('off')

            # Save the plot to the specified output path
            if output_path:
                plt.savefig(output_path)

            plt.show()
            plt.close()


        return donut_mask

    # #Run functions to generate donut masks:
    cytosol_donut_mask = generate_donut_mask(masks_cytosol, n=20, plot=True, output_path=os.path.join(output_directory, f'{image_name}_cytosol_donut.png'))
    nuclei_donut_mask = generate_donut_mask(masks_nuclei, n=10, plot=True, output_path=os.path.join(output_directory, f'{image_name}_nuclei_donut.png'))


#### 2.2 mRNA-membrane colocalization

In [ ]:
#Code to visualize colocalization with membranes

if calculate_membrane_colocalization:
    def calculate_membrane_colocalization(coord, rna_max, output_directory, rna_channel, image_name, plot=True):
        """
        Calculate and optionally plot the colocalization of RNA spots with membranes.

        Parameters:
            coord: np.ndarray
                The coordinates of RNA spots.
            rna_max: np.ndarray
                Maximum projection of the RNA channel for plotting.
            output_directory: str
                Directory to save the output plots.
            rna_channel: str
                Name of the RNA channel (e.g., "mCherry").
            image_name: str
                Name of the image being processed.
            plot: bool
                Whether to create and save plots.

        Returns:
            tuple: (spots_in_membranes, spots_out_membranes)
        """
       
        mask = cytosol_donut_mask  # Cytosol/membrane mask
        ndim = 3  # 3D data

        # Identify objects in regions
        spots_in_membranes, spots_out_membranes = bigfish.multistack.identify_objects_in_region(mask, coord, ndim)

        # Log the results
        print(f'{rna_name} spots detected (on membranes):')
        print(f"  Shape: {spots_in_membranes.shape}")
        print(f"  Dtype: {spots_in_membranes.dtype}\n")
        print(f'{rna_name} spots detected (in cytosol):')
        print(f"  Shape: {spots_out_membranes.shape}")
        print(f"  Dtype: {spots_out_membranes.dtype}")


        # Plot detected spots on membranes
        if plot:
            # Plot detected spots on membranes
            in_membranes_output = os.path.join(output_directory, rna_channel + '_in_membranes_' + image_name + '.png')
            plot.plot_detection(
                mask,
                spots=spots_in_membranes,
                radius=1,
                color="red",
                path_output=in_membranes_output,
                title=f'Red = {rna_name} on membranes',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

            # Plot detected spots in cytosol
            out_membranes_output = os.path.join(output_directory, rna_channel + '_out_membranes_' + image_name + '.png')
            plot.plot_detection(
                rna_max,
                spots=spots_out_membranes,
                radius=1,
                color="blue",
                path_output=out_membranes_output,
                title=f'Blue = {rna_name} in cytosol',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

            # Combined plot
            combined_membranes_output = os.path.join(output_directory, rna_channel + '_combined_membranes_output_' + image_name + '.png')
            plot.plot_detection(
                rna_max,
                spots=[spots_in_membranes, spots_out_membranes],
                radius=2,
                color=["red", "blue"],
                path_output=combined_membranes_output,
                title=f'Red = {rna_name} on membrane | Blue = {rna_name} mRNA in cytosol',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )

        
        # Quantification: number of spots in membranes and cytoplasm
        num_spots_in_membranes = spots_in_membranes.shape[0]
        # num_spots_in_cyto = spots_out_membranes.shape[0]

        # Add new columns with default values (e.g., NaN)
        df_quantification[f'{rna_name} in membranes'] = np.nan
        # df_quantification['mCherry_out_membranes'] = np.nan

        # Update the new columns with the counts for the specific image ID
        df_quantification.loc[df_quantification['Image ID'] == image_name, f'{rna_name} in membranes'] = num_spots_in_membranes
        # df_quantification.loc[df_quantification['Image ID'] == image_name, 'mCherry_out_membranes'] = num_spots_in_cyto

        # Display the updated DataFrame
        df_quantification
        
        
        return spots_in_membranes, spots_out_membranes

    # Call the function for both RNA channels
    rna_names = [Cy5, mCherry]  # Replace with your actual mRNA names
    rna_images = [image_Cy5, image_mCherry]  # Replace with your actual 2D/3D RNA images
    rna_coords = [ch0_array, ch1_array]  # Coordinates for each RNA channel

    for rna_name, rna_image, coord in zip(rna_names, rna_images, rna_coords):
        spots_in_membranes, spots_out_membranes = calculate_membrane_colocalization(
            coord=coord,  # Use the specific coordinates for each RNA channel
            rna_max=rna_image,  # Use the generic rna_max variable for the RNA image
            output_directory=output_directory,
            rna_channel=rna_name,  # Pass the RNA name (mCherry or Cy5)
            image_name=image_name,
            plot=plot
        )
    # # Display the updated DataFrame
    df_quantification

#### 2.3 Distance from membrane analysis

In [ ]:
# #Erode masks and calculate layers from the membrane
def generate_inner_concentric_layers(original_mask, n_layers=5, spacing=2, plot=False, output_path=None):
    layered_mask = np.zeros_like(original_mask, dtype=np.uint8)

    for label in np.unique(original_mask):
        if label == 0:
            continue

        single_cell = (original_mask == label)
        current_mask = single_cell.copy()

        for layer in range(1, n_layers + 1):
            eroded = binary_erosion(current_mask, disk(spacing))
            ring = current_mask & ~eroded
            layered_mask[ring] = layer
            current_mask = eroded
            if not np.any(current_mask):
                break

    if plot:
        # Create a discrete colormap from the rainbow base
        base_cmap = plt.colormaps['rainbow']
        discrete_colors = base_cmap(np.linspace(0, 1, n_layers))
        discrete_cmap = ListedColormap(discrete_colors)
        discrete_cmap.set_bad(color='black')  # Background to black

        # --------- Combined layered plot ---------
        masked = np.ma.masked_where(layered_mask == 0, layered_mask)
        fig_combined, ax_combined = plt.subplots(figsize=(6, 5))
        im = ax_combined.imshow(masked, cmap=discrete_cmap, vmin=1, vmax=n_layers)
        ax_combined.set_title('Inner Concentric Layers')
        ax_combined.axis('off')

        cbar = fig_combined.colorbar(im, ax=ax_combined, ticks=np.arange(1, n_layers + 1))
        cbar.set_label('Layer')
        cbar.ax.set_yticklabels([str(i) for i in range(1, n_layers + 1)])

        if output_path is not None:
            base, ext = os.path.splitext(output_path)
            combined_path = f"{base}_combined{ext}"
            fig_combined.savefig(combined_path, bbox_inches='tight', dpi=300)

        plt.show()
        plt.close()


        # --------- Individual layer subplots ---------
        fig_layers, axes = plt.subplots(1, n_layers, figsize=(4 * n_layers, 4))
        for i in range(n_layers):
            ax = axes[i]
            layer_num = i + 1
            single_layer = np.ma.masked_where(layered_mask != layer_num, layered_mask)
            im = ax.imshow(single_layer, cmap=discrete_cmap, vmin=1, vmax=n_layers)
            ax.set_title(f'Layer {layer_num}', color='black')
            ax.axis('off')
            ax.set_facecolor('black')

        plt.tight_layout()

        if output_path is not None:
            layers_path = f"{base}_individual{ext}"
            fig_layers.savefig(layers_path, bbox_inches='tight', dpi=300)

        plt.show()
        plt.close()


    return layered_mask

if run_concentric_layers_analysis:
    layered_mask = generate_inner_concentric_layers(
        original_mask=masks_cytosol,
        n_layers=6,
        spacing=25,
        plot=True,
        output_path=os.path.join(output_directory, f'{image_name}_concentric_layers.png')
    )


In [ ]:
# # PLot mRNA abundance on concentr

def plot_rna_on_individual_layers(rna_names, rna_images, rna_coords, layered_mask, image_name, output_directory=None, plot=True, grid_width=80, grid_height=80):
    n_layers = len(np.unique(layered_mask)) - 1  # Exclude background (layer 0)

    for rna_name, rna_image, coord in zip(rna_names, rna_images, rna_coords):
        y_coords = coord[:, 1]
        x_coords = coord[:, 2]
        y_coords = np.clip(y_coords, 0, layered_mask.shape[0] - 1)
        x_coords = np.clip(x_coords, 0, layered_mask.shape[1] - 1)

        # Build heatmaps and determine global vmin/vmax
        layer_heatmaps = []
        global_max = 0
        global_min = np.inf

        for i in range(n_layers):
            layer_num = i + 1
            single_layer_mask = (layered_mask == layer_num)
            valid_idx = single_layer_mask[y_coords, x_coords]
            layer_spots_x = x_coords[valid_idx]
            layer_spots_y = y_coords[valid_idx]

            grid = np.zeros((grid_height, grid_width), dtype=int)
            cell_width = layered_mask.shape[1] / grid_width
            cell_height = layered_mask.shape[0] / grid_height

            for x, y in zip(layer_spots_x, layer_spots_y):
                cell_x = int(x / cell_width)
                cell_y = int(y / cell_height)
                if 0 <= cell_x < grid_width and 0 <= cell_y < grid_height:
                    grid[cell_y, cell_x] += 1

            heatmap_resized = zoom(grid, (layered_mask.shape[0] / grid_height, layered_mask.shape[1] / grid_width), order=0)
            heatmap_resized = np.ma.masked_where(~single_layer_mask, heatmap_resized)
            layer_heatmaps.append((heatmap_resized, single_layer_mask))

            if np.any(heatmap_resized):
                global_min = min(global_min, heatmap_resized.min())
                global_max = max(global_max, heatmap_resized.max())

        # Create figure with constrained layout (avoids tight_layout warning)
        fig = plt.figure(figsize=(4 * n_layers + 1, 4), facecolor='black', constrained_layout=True)
        gs = gridspec.GridSpec(1, n_layers + 1, figure=fig, width_ratios=[1]*n_layers + [0.05], wspace=0.05)
        axes = [fig.add_subplot(gs[0, i]) for i in range(n_layers)]

        for i in range(n_layers):
            ax = axes[i]
            heatmap_masked, single_layer_mask = layer_heatmaps[i]

            im = ax.imshow(heatmap_masked, cmap='CMRmap', interpolation='nearest', alpha=0.9, vmin=global_min, vmax=global_max)
            ax.contour(single_layer_mask.astype(float), levels=[0.5], colors='white', linewidths=0.8)
            ax.set_title(f'{rna_name} - Layer {i + 1}', color='white')
            ax.axis('off')
            ax.set_facecolor('black')

        # Colorbar
        cbar_ax = fig.add_subplot(gs[0, -1])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label('mRNA density', color='white')
        cbar.ax.tick_params(colors='white')

        fig.patch.set_facecolor('black')

        # Always save if output_directory is set
        if output_directory:
            output_path = os.path.join(output_directory, f'{image_name}_{rna_name}_layered_heatmaps.png')
            plt.savefig(output_path, facecolor='black', bbox_inches='tight')

        if plot:
            plt.show()
        else:
            plt.close()
# Example call to the function
rna_names = [Cy5, mCherry]  # Replace with your actual mRNA names
rna_images = [image_Cy5, image_mCherry]  # Replace with your actual 2D/3D RNA images
rna_coords = [ch0_array, ch1_array]  # Coordinates for each RNA channel

if run_concentric_layers_analysis:
    plot_rna_on_individual_layers(
        rna_names=rna_names,  # RNA channel names
        rna_images=rna_images,  # Maximum projections of RNA channels
        rna_coords=rna_coords,  # Coordinates for each RNA channel
        layered_mask=layered_mask,  # Cytosolic layers mask
        image_name=image_name,  # Image identifier
        output_directory=output_directory  # Directory to save the plots
    )

In [ ]:
## Plot the total abundance per normalized layer
def plot_rna_abundance_by_layer(rna_names, rna_coords, layered_mask, image_name, output_directory, plot=True):
    n_layers = len(np.unique(layered_mask)) - 1
    layer_ids = np.arange(1, n_layers + 1)
    data_dict = {"Layer": layer_ids}

    fig, ax = plt.subplots(figsize=(8, 5))

    for rna_name, coord in zip(rna_names, rna_coords):
        y_coords = np.clip(coord[:, 1], 0, layered_mask.shape[0] - 1)
        x_coords = np.clip(coord[:, 2], 0, layered_mask.shape[1] - 1)

        layer_labels = layered_mask[y_coords, x_coords]
        layer_labels = layer_labels[layer_labels > 0]

        abundance = np.array([(layer_labels == i).sum() for i in layer_ids])
        data_dict[rna_name] = abundance
        ax.plot(layer_ids, abundance, marker='o', label=rna_name)

    df = pd.DataFrame(data_dict)

    ax.set_facecolor('white')
    ax.set_xlabel('Layer (distance from membrane)', color='black')
    ax.set_ylabel('Total mRNA abundance', color='black')
    ax.set_title(f'mRNA Abundance by Layer: {image_name}', color='black')
    ax.tick_params(colors='black')
    ax.legend()
    plt.grid(True, color='gray', linestyle='--', linewidth=0.5, axis='y', alpha=0.5)
    fig.patch.set_facecolor('white')
    plt.xticks(layer_ids)

    if output_directory:
        plot_path = os.path.join(output_directory, f'{image_name}_total_abundance_per_layer.png')
        csv_path = os.path.join(output_directory, f'{image_name}_total_abundance_per_layer.csv')
        plt.savefig(plot_path, facecolor='white', bbox_inches='tight')
        df.to_csv(csv_path, index=False)

    
    if plot:
        plt.show()
    else:
        plt.close()


## Plot the relative abundance per normalized layer

def plot_relative_rna_abundance_by_layer(rna_names, rna_coords, layered_mask, image_name, output_directory, plot=True):
    n_layers = len(np.unique(layered_mask)) - 1
    layer_ids = np.arange(1, n_layers + 1)
    data_dict = {"Layer": layer_ids}

    fig, ax = plt.subplots(figsize=(8, 5))

    for rna_name, coord in zip(rna_names, rna_coords):
        y_coords = np.clip(coord[:, 1], 0, layered_mask.shape[0] - 1)
        x_coords = np.clip(coord[:, 2], 0, layered_mask.shape[1] - 1)

        layer_labels = layered_mask[y_coords, x_coords]
        layer_labels = layer_labels[layer_labels > 0]

        abundance = np.array([(layer_labels == i).sum() for i in layer_ids])
        relative = abundance / abundance.sum() * 100 if abundance.sum() > 0 else np.zeros_like(abundance)
        data_dict[rna_name] = relative
        ax.plot(layer_ids, relative, marker='o', label=rna_name)

    df = pd.DataFrame(data_dict)

    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    ax.set_xlabel('Layer (distance from membrane)', color='black')
    ax.set_ylabel('Relative mRNA abundance (%)', color='black')
    ax.set_title(f'Relative mRNA Abundance by Layer: {image_name}', color='black')
    ax.tick_params(colors='black')
    ax.legend()
    plt.xticks(layer_ids)
    ax.grid(True, color='gray', linestyle='--', linewidth=0.5, axis='y', alpha=0.5)

    
    if output_directory:
        plot_path = os.path.join(output_directory, f'{image_name}_relative_abundance_per_layer.png')
        csv_path = os.path.join(output_directory, f'{image_name}_relative_abundance_per_layer.csv')
        plt.savefig(plot_path, facecolor='white', bbox_inches='tight')
        df.to_csv(csv_path, index=False)

    if plot:
        plt.show()
    else:
        plt.close()

# #Call plotting functions
if run_concentric_layers_analysis:
    plot_rna_abundance_by_layer(
        rna_names=rna_names,         # list of RNA channel names (e.g., ['Cy5', 'mCherry'])
        rna_coords=rna_coords,       # list of coordinate arrays (e.g., [ch0_array, ch1_array])
        layered_mask=layered_mask,   # your segmented layered mask (2D array)
        image_name=image_name,
        output_directory=output_directory # string identifying the image (for plot title)
    )


    plot_relative_rna_abundance_by_layer(
        rna_names=rna_names,         # list of RNA channel names (e.g., ['Cy5', 'mCherry'])
        rna_coords=rna_coords,       # list of coordinate arrays (e.g., [ch0_array, ch1_array])
        layered_mask=layered_mask,   # your segmented layered mask (2D array)
        image_name=image_name,
        output_directory=output_directory# string identifying the image (for plot title)
    )



### 3. mRNA-p granule colocalization analysis

In [ ]:
#p-granule mask for co-localization

image_pgranules = image_FITC
image_pgranules = gaussian_filter(image_pgranules, sigma=0.2)

if generate_pgranule_mask:
    def mask_pgranule(image_pgranules):
        MIN_CELL_SIZE = 5
        list_masks_pgranules = []
        list_thresholds = np.arange(0.4, 0.95, 0.02)
        array_number_detected_masks = np.zeros(len(list_thresholds))
        for i,tested_ts in enumerate(list_thresholds):
            image_pgranules_binary = image_pgranules.copy()
            max_value_image = np.max(image_pgranules_binary)
            image_pgranules_binary[image_pgranules_binary < max_value_image*tested_ts] = 0.5
            image_pgranules_binary[image_pgranules_binary > max_value_image*tested_ts] = 1
            labels = measure.label(image_pgranules_binary)
            filtered_labels = morphology.remove_small_objects(labels, min_size=MIN_CELL_SIZE)
            unique_filtered_labels = np.unique(filtered_labels)
            tested_masks_pgranules = np.zeros_like(filtered_labels)
            for idx, old_label in enumerate(unique_filtered_labels):
                tested_masks_pgranules[filtered_labels == old_label] = idx
            list_masks_pgranules.append(tested_masks_pgranules)
            array_number_detected_masks[i]= metric_max_cells_and_area( tested_masks_pgranules) 
        selected_index = np.argmax(array_number_detected_masks)
        masks_pgranules = list_masks_pgranules [selected_index]
        return masks_pgranules

    masks_pgranules = mask_pgranule(image_pgranules)

    # Plotting
    color_map = 'Greys_r'
    fig, ax = plt.subplots(1,2, figsize=(10, 4))
    # Plotting the heatmap of a section in the image
    ax[0].imshow(image_pgranules,cmap=color_map)
    ax[1].imshow(masks_pgranules,cmap=color_map, alpha=0.9)
    ax[0].set(title='p granules'); ax[0].axis('off');ax[0].grid(False)
    ax[1].set(title='mask p granules'); ax[1].axis('off');ax[1].grid(False)

    # Save the figure
    output_path = os.path.join(output_directory, mCherry + '_pgranule_mask_' + image_name + '.png')

    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1)
    

In [ ]:
# #generate p granule masks

if generate_pgranule_mask:
    def mask_pgranule_within_cytosol(image_pgranules, masks_cytosol, generate_pgranule_mask=True):
        MIN_CELL_SIZE = 5
        list_masks_pgranules = []
        list_thresholds = np.arange(0.5, 0.98, 0.002)
        array_number_detected_masks = np.zeros(len(list_thresholds))

        # Apply a Gaussian filter to reduce noise and smooth the image
        image_pgranules = gaussian(image_pgranules, sigma=0.5)  # Adjust sigma if needed
        image_pgranules = median(image_pgranules, np.ones((3, 3)))  # Optional median filter

        # Mask image_pgranules with the cytosol mask to restrict processing to cytosol area
        image_pgranules_masked = image_pgranules * masks_cytosol

        for i, tested_ts in enumerate(list_thresholds):
            image_pgranules_binary = image_pgranules_masked.copy()

            # Apply adaptive thresholding (Local thresholding) only on masked region
            block_size = 35  # Size of the local window for thresholding
            adaptive_threshold = threshold_local(image_pgranules_binary, block_size, offset=10)  # Adjust offset for sensitivity

            # Create a binary image using local thresholding
            image_pgranules_binary = image_pgranules_binary > adaptive_threshold

            # Optional: further thresholding based on tested_ts multiplier
            max_value_image = np.max(image_pgranules_binary)
            image_pgranules_binary[image_pgranules_binary < max_value_image * tested_ts] = 0
            image_pgranules_binary[image_pgranules_binary >= max_value_image * tested_ts] = 1

            # Label connected components
            labels = measure.label(image_pgranules_binary)

            # Remove small objects
            filtered_labels = morphology.remove_small_objects(labels, min_size=MIN_CELL_SIZE)
            unique_filtered_labels = np.unique(filtered_labels)

            # Create the masks for the detected p-granules
            tested_masks_pgranules = np.zeros_like(filtered_labels)
            for idx, old_label in enumerate(unique_filtered_labels):
                tested_masks_pgranules[filtered_labels == old_label] = idx

            list_masks_pgranules.append(tested_masks_pgranules)
            array_number_detected_masks[i] = metric_max_cells_and_area(tested_masks_pgranules)

        # Select the best threshold based on the number of detected masks
        selected_index = np.argmax(array_number_detected_masks)
        masks_pgranules = list_masks_pgranules[selected_index]

        return masks_pgranules

    # Example usage:
    image_pgranules = image_FITC  # Your input image here
    masks_pgranules = mask_pgranule_within_cytosol(image_pgranules, masks_cytosol)

    # Plotting
    color_map = 'Greys_r'
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(image_pgranules, cmap=color_map)
    ax[1].imshow(masks_pgranules, cmap=color_map, alpha=0.9)
    ax[0].set(title='p granules'); ax[0].axis('off'); ax[0].grid(False)
    ax[1].set(title='mask p granules'); ax[1].axis('off'); ax[1].grid(False)

    # Save the figure
    output_path = os.path.join(output_directory, mCherry + '_pgranule_mask_' + image_name + '.png')
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1)


In [ ]:
# # Using cellpose models to mask pgrnaules. challenching because the spots are small and close. cyto2 doesnt perfomr very well. 
run_pgranule_segmentation = False

if run_pgranule_segmentation:
    cytosol_image = bf[..., 0] if bf.ndim == 3 else bf
    FITC_image = image_FITC[..., 2] if image_FITC.ndim == 3 else image_FITC

    # Run Cellpose for P granule segmentation
    model_pgranule = models.Cellpose(model_type='cyto2')
    masks_pgranule, flows_pgranule, styles_pgranule, diams_pgranule = model_pgranule.eval(
        FITC_image, diameter=15, channels=[0, 0],
        cellprob_threshold=0.0, flow_threshold=0.7
    )

    outlines_cytosol = utils.outlines_list(masks_cytosol)  # Presumed to be defined earlier
    outlines_pgranule = utils.outlines_list(masks_pgranule)

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))

    ax[0].imshow(cytosol_image, cmap='gray')
    for o in outlines_cytosol:
        ax[0].plot(o[:, 0], o[:, 1], color='lime', linewidth=1)
    ax[0].set_title('Embryo Mask')
    ax[0].axis('off')

    ax[1].imshow(FITC_image, cmap='gray')
    for o in outlines_pgranule:
        ax[1].plot(o[:, 0], o[:, 1], color='deepskyblue', linewidth=1)
    ax[1].set_title('Pgranule Mask')
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close()


    # save
    #plt.savefig(os.path.join(output_directory, "pgranule_segmentation.png"), dpi=300, bbox_inches='tight')


In [ ]:
#p-granule mask for co-localization

if calculate_pgranule_colocalization:
    def calculate_pgranule_colocalization(coord, rna_max, masks_pgranules, output_directory, rna_channel, image_name, plot=True):
        """
        Calculate and optionally plot the colocalization of RNA spots with p-granules.

        Parameters:
            coord: np.ndarray
                The coordinates of RNA spots.
            rna_max: np.ndarray
                Maximum projection of the RNA channel for plotting.
            masks_pgranules: np.ndarray
                Mask of the p-granule region.
            output_directory: str
                Directory to save the output plots.
            rna_channel: str
                Name of the RNA channel (e.g., "Cy5").
            image_name: str
                Name of the image being processed.

        Returns:
            tuple: (spots_in_pgranules, spots_out_pgranules)
        """
        ndim = 3  # 3D data

        # Identify objects in p-granules
        spots_in_pgranules, spots_out_pgranules = bigfish.multistack.identify_objects_in_region(masks_pgranules, coord, ndim)

        # Log the results
        print(f'{rna_channel} spots detected (in p-granules):')
        print(f"  Shape: {spots_in_pgranules.shape}")
        print(f"  Dtype: {spots_in_pgranules.dtype}\n")
        

        if plot:
            # Plot spots in p-granules
            in_pgranules_output = os.path.join(output_directory, rna_channel + '_in_pgranules_' + image_name + '.png')
            plot.plot_detection(
                rna_max,
                spots=spots_in_pgranules,
                radius=1,
                color="red",
                path_output=in_pgranules_output,
                title=f'Red = {rna_channel} in p-granules',
                linewidth=3,
                contrast=True,
                framesize=(10, 5)
            )


        # Quantification: number of spots in p-granules and outside p-granules
        num_spots_in_pgranules = spots_in_pgranules.shape[0]

        # Add new columns with default values (e.g., NaN)
        df_quantification[f'{rna_channel} in pgranules'] = np.nan

        # Update the new columns with the counts for the specific image ID
        df_quantification.loc[df_quantification['Image ID'] == image_name, f'{rna_channel} in pgranules'] = num_spots_in_pgranules

        return spots_in_pgranules, spots_out_pgranules

    # Define RNA channels and other variables
    rna_names = [Cy5, mCherry]  # Replace with your actual mRNA names
    rna_images = [image_Cy5, image_mCherry]  # Replace with your actual 2D/3D RNA images
    rna_coords = [ch0_array, ch1_array]  # Coordinates for each RNA channel

    # Call the function for each RNA channel
    for rna_name, rna_image, coord in zip(rna_names, rna_images, rna_coords):
        spots_in_pgranules, spots_out_pgranules = calculate_pgranule_colocalization(
            coord=coord,
            rna_max=rna_image,
            masks_pgranules=masks_pgranules,
            output_directory=output_directory,
            rna_channel=rna_name,
            image_name=image_name,
            plot=plot  # Pass the boolean flag for plotting
        )

    # Display the updated DataFrame
    display(df_quantification)

### 4. mRNA-mRNA colocalization (FLARIM - translating mRNAs)

In [ ]:
# #Calculate mRNA-mRNA colocalization

if calculate_mRNA_mRNA_colocalization:
    def mRNA_coloc(ch0_array, ch1_array, voxel_size, image_cy5, image_mCherry, image_name, Cy5, mCherry, plot):
        """
        Detect colocalized mRNA spots between two channels with flexibility for proximity-based matching.
        """
        # Set proximity threshold in voxel space
        threshold = 5  # Adjust as needed (in pixels or voxel space)

        # Convert coordinates to nanometers (or keep as is if already in correct units)
        ch0_scaled = ch0_array * voxel_size
        ch1_scaled = ch1_array * voxel_size

        # Build KDTree for fast nearest-neighbor search
        tree = cKDTree(ch1_scaled)

        # Find nearest neighbors for spots in ch0_array
        distances, indices = tree.query(ch0_scaled, distance_upper_bound=threshold)

        # Filter out matches that exceed the threshold
        valid_matches = distances < threshold
        colocalized_spots_ch0 = ch0_array[valid_matches]
        colocalized_spots_ch1 = ch1_array[indices[valid_matches]]

        # Print summary of colocalized spots
        print(f"Total {Cy5} spots: {ch0_array.shape[0]}")
        print(f"Total {mCherry} spots: {ch1_array.shape[0]}")
        print(f"Colocalized spots: {colocalized_spots_ch0.shape[0]}")

        # Define output filename
        mRNA_mRNA_output = os.path.join(output_directory, f"mRNA_coloc_mRNA_{image_name}.png")

        # Plot and save the colocalization results
        plot.plot_detection(
            image_mCherry,
            spots=[colocalized_spots_ch0, colocalized_spots_ch1], 
            radius=2, 
            path_output=mRNA_mRNA_output,
            color=["red", "purple"],
            title=f"Red = {Cy5} | Purple = {mCherry}",
            linewidth=3, contrast=True, framesize=(10, 5)
        )

        # Return colocalized spots and distances for further analysis
        return colocalized_spots_ch0, colocalized_spots_ch1, distances[valid_matches]

    # Call the function
    colocalized_spots_ch0, colocalized_spots_ch1, distances = mRNA_coloc(
        ch0_array, ch1_array, voxel_size, image_Cy5, image_mCherry, image_name, Cy5, mCherry, plot
    )


### 5. Log calculation of mRNA vs protein

In [1]:
# # Heatmap of protein levels with intensity calculations restricted to the mask
# import matplotlib.pyplot as plt
# fitc_projection = image_FITC

# def protein_expression(intensity_image, mask, output_filename, title='Protein Expression Heatmap', grid_width=80, grid_height=80, threshold=0.7):
#     """
#     Generates a protein expression heatmap, restricting intensity calculations to the mask and applying a threshold 
#     to remove background noise.

#     Parameters:
#     - intensity_image: 2D numpy array of protein intensity values
#     - mask: 2D numpy array of the mask where protein expression is considered
#     - output_filename: Path to save the resulting heatmap image
#     - title: Title of the heatmap
#     - grid_width: Number of grid columns for analysis
#     - grid_height: Number of grid rows for analysis
#     - threshold: Intensity threshold (0 to 1) to remove background noise
#     """
    
#     # Apply the threshold to remove background noise (similar to your nuclear_segmentation)
#     max_value_image = np.max(intensity_image)
#     intensity_image_thresholded = intensity_image.copy()
#     intensity_image_thresholded[intensity_image_thresholded < max_value_image * threshold] = 0

#     # Calculate the width and height of each grid cell
#     img_width, img_height = intensity_image.shape[1], intensity_image.shape[0]
#     cell_width = img_width / grid_width
#     cell_height = img_height / grid_height

#     # Create an empty grid to store the mean intensities
#     grid = np.zeros((grid_height, grid_width), dtype=float)

#     # Populate the grid with mean intensities within the mask
#     for i in range(grid_height):
#         for j in range(grid_width):
#             # Determine the boundaries of the current grid cell
#             x_start = int(j * cell_width)
#             x_end = int((j + 1) * cell_width)
#             y_start = int(i * cell_height)
#             y_end = int((i + 1) * cell_height)

#             # Extract the subregion of the intensity image and the mask
#             region_intensity = intensity_image_thresholded[y_start:y_end, x_start:x_end]
#             region_mask = mask[y_start:y_end, x_start:x_end]

#             # Calculate the mean intensity only inside the mask
#             if np.any(region_mask):
#                 grid[i, j] = np.mean(region_intensity[region_mask > 0])
#             else:
#                 grid[i, j] = np.nan  # Set to NaN if no mask is present in the region

#     # Create a colormap that handles NaN values by setting them to black
#     cmap = plt.get_cmap('CMRmap').copy()
#     cmap.set_bad(color='black')

#     # Create a heatmap, handling NaN values by setting them to black
#     masked_grid = np.ma.masked_invalid(grid)  # Mask NaN values
#     plt.imshow(masked_grid, cmap=cmap, interpolation='nearest', vmin=np.nanmin(grid), vmax=np.nanmax(grid))
#     plt.title(title)
#     plt.axis('off')

#     # Create a vertical color bar
#     cbar = plt.colorbar(orientation='vertical', shrink=0.5)  # Adjust the shrink parameter to make it smaller
#     cbar.ax.text(1, 1.05, 'Higher\nExpression', transform=cbar.ax.transAxes, ha='center')
#     cbar.ax.text(1, -0.19, 'Lower\nExpression', transform=cbar.ax.transAxes, ha='center')
#     cbar.set_ticks([])

#     # Save the heatmap
#     plt.savefig(output_filename)
#     plt.show()
#     plt.close()


# # Example usage with mask and intensity projection, including threshold for background noise removal
# if fitc_projection is not None and masks_cytosol is not None:
#     protein_expression(fitc_projection, masks_cytosol, os.path.join(output_directory, 'FITC_protein_expression_heatmap.png'), threshold=0.6)


In [ ]:
# def apply_threshold(image, threshold):
#     """
#     Apply thresholding to the input image, setting values below threshold to 0.
#     """
#     max_value_image = np.max(image)
#     binary_image = np.zeros_like(image)
#     binary_image[image >= max_value_image * threshold] = 1
#     return binary_image

# def log_contrast_with_white_background(mCherry_heatmap, fitc_projection, masks_cytosol, threshold=0.7, cmap='PRGn', grid_width=80, grid_height=80):
#     from skimage.transform import resize
#     import numpy as np
#     import matplotlib.pyplot as plt

#     # Resize the cytosol mask to match the mCherry_heatmap dimensions
#     masks_cytosol_resized = resize(masks_cytosol, mCherry_heatmap.shape, anti_aliasing=True)
    
#     # Apply thresholding
#     mcherry_thresholded = mCherry_heatmap * apply_threshold(mCherry_heatmap, threshold)
#     fitc_thresholded = fitc_projection * apply_threshold(fitc_projection, threshold)

#     # Apply mask
#     mcherry_masked = mcherry_thresholded * masks_cytosol_resized
#     fitc_masked = fitc_thresholded * masks_cytosol_resized

#     # Initialize log contrast map
#     log_contrast_map = np.full(mCherry_heatmap.shape, np.nan)

#     # Compute contrast where mask is > 0
#     mask_inside = masks_cytosol_resized > 0
#     if np.any(mask_inside):
#         log_contrast_map[mask_inside] = np.log1p(mcherry_masked[mask_inside]) - np.log1p(fitc_masked[mask_inside])

#     # Mask invalid values and set white background
#     masked_contrast_map = np.ma.masked_invalid(log_contrast_map)
#     cmap = plt.get_cmap('CMRmap').copy()
#     cmap.set_bad(color='black')

#     # Plot
#     plt.imshow(masked_contrast_map, cmap=cmap, interpolation='nearest', 
#                vmin=np.nanmin(log_contrast_map), vmax=np.nanmax(log_contrast_map))
#     plt.colorbar(label='Log Contrast')
#     plt.title('Log Contrast between mCherry and FITC')
#     plt.axis('off')
#     plt.tight_layout()
#     plt.show()
#     plt.close()

#     return log_contrast_map
# # Example usage with higher grid resolution for better color granularity
# fitc_resized = resize(fitc_projection, mCherry_heatmap.shape, anti_aliasing=True)
# log_contrast_map = log_contrast_with_white_background(mCherry_heatmap, fitc_resized, masks_cytosol, threshold=0.3, grid_width=40, grid_height=40)


### 6. Export data report

In [2]:
# #pdf report

output_pdf_path = os.path.join(output_directory, "report.pdf")


# Collecting all images and csvs into a list and sorting based on creation time
output_file_paths = []
for filename in os.listdir(output_directory):
    if filename.lower().endswith((".png", ".csv")):
        fullPath = os.path.join(output_directory, filename)
        output_file_paths.append(fullPath)
sortedImages = sorted(output_file_paths, key=lambda image: os.path.getctime(image))


#Collecting current date for report generation statement
runTime = datetime.now()
currentDate = runTime.date()


# How to handle csvs function using platypus for text based csv file
def csvFunc(file, c, margin, origin, padding):
    csvTitle = os.path.basename(file)
    #Reads csv data
    with open(file, newline="") as csvFile:
        reader = csv.reader(csvFile)
        data = list(reader)
    #Creates a table from csv data
    table = Table(data)
    table.setStyle(TableStyle([
        ("TEXTCOLOR",    (0,0), (-1,0),  colors.white),
        ("BACKGROUND",   (0,0), (-1,0),  colors.grey),
        ("ALIGN",        (0,0), (-1,-1), "CENTER"),
        ("FONTNAME",     (0,0), (-1,0),  "Times-Roman"),
        ("BOTTOMPADDING",(0,0), (-1,0),  12),
        ("BACKGROUND",   (0,1), (-1,-1), colors.beige),
        ("GRID",         (0,0), (-1,-1), 0.5, colors.black)
    ]))
    #Get the width and height of the table
    tableWidth, tableHeight = table.wrapOn(c, 400, 600)
    #Adjust origin for the new file
    origin -= tableHeight + padding
    #Draw table
    table.drawOn(c, margin, origin)
    return(origin, csvTitle)


# How to handle images function
def imageFunc(file):
    imageTitle = os.path.basename(file)
    with Image.open(file) as img:
        width, height = img.size
        aspectRatio = width / height
    width = 145.6 * aspectRatio
    if width > 548:
        height = 548 / aspectRatio
        width = 548
    else:
        height = 145.6
    return(height, width, imageTitle)


# Adding page numbers
def addPageNumber(c):
    pageNum = c.getPageNumber()
    text = f"{pageNum}"
    c.setFont("Times-Roman", 10)
    c.drawRightString(580, 32, text)


# Making the canvas and adding the header info
    # letter = max Y coord=792, and max X coord=612
margin = 32
c = canvas.Canvas(output_pdf_path, pagesize=letter)
c.setFont("Times-Roman", 16)
c.drawString(margin, 728, f"{image_name}")
c.setFont("Times-Roman", 14)
c.drawString(margin, 713, f"Report Generated: {currentDate}")


# Formatting and drawing loop for csvs, images, and their titles
origin = 700
padding = 20
for file in sortedImages:
    if file.endswith(".png"):
        height, width, imageTitle = imageFunc(file)
        if origin >= height + padding + 15:
            c.setFont("Times-Roman", 12)
            c.drawString(margin, origin - 15, f"{imageTitle}")
            c.drawImage(file, margin, origin - height - padding, width, height)
            origin -= height + padding + 10
        else:
            c.showPage()
            c.setFont("Times-Roman", 16)
            c.drawString(margin, 728, f"{image_name}")
            origin = 710
            c.setFont("Times-Roman", 12)
            c.drawString(margin, origin, f"{imageTitle}")
            c.drawImage(file, margin, origin - height - padding, width, height)
            origin -= height + padding + 20
            addPageNumber(c)
    elif file.endswith(".csv"):
        origin, csvTitle = csvFunc(file, c, margin, origin, padding)


# # Try to dynamically find and use the first .out file in the directory
# for outputFile in os.listdir(output_directory):
#     if outputFile.lower().endswith(".out"):
#         outFile = os.path.join(output_directory, outputFile)
#         outFileName = os.path.basename(outputFile)

#         # If file exists, read and add to report
#         if os.path.isfile(outFile):
#             textData = c.beginText(margin, origin - padding)
#             with open(outFile, 'r') as file:
#                 for line in file:
#                     textData.textLine(line.strip())
#             c.drawString(margin, origin, f"{outFileName}")
#             c.drawText(textData)
#         break  # Stop after processing the first .out file


# Dynamically find and use the first .out file in the directory
for outputFile in os.listdir(output_directory):
    if outputFile.lower().endswith(".out"):
        outFile = os.path.join(output_directory, outputFile)
        outFileName = os.path.basename(outputFile)

        if os.path.isfile(outFile):
            # Estimate number of lines
            with open(outFile, 'r') as f:
                lines = f.readlines()

            # Approximate space needed (12 pts per line)
            required_space = len(lines) * 12 + 40

            if origin < required_space:
                c.showPage()
                origin = 720
                c.setFont("Times-Roman", 16)
                c.drawString(margin, 728, f"{image_name}")
                c.setFont("Times-Roman", 12)

            # Add log file name
            c.setFont("Times-Roman", 12)
            c.drawString(margin, origin, f"{outFileName}")
            origin -= 16

            # Write lines
            textData = c.beginText(margin, origin)
            textData.setFont("Times-Roman", 10)
            for line in lines:
                textData.textLine(line.strip())
            c.drawText(textData)

            # Update origin
            origin -= len(lines) * 12 + padding

        break  # Stop after processing the first .out file



c.save()
print(f"PDF report created: {output_pdf_path}")

NameError: name 'os' is not defined